# ML-06 — Signal Audit

This notebook checks whether May search signals are associated with the observed June decline label.


## 1. Distributions

This audit uses the May feature window and the observed June decline label from the same eligible pages as Week 3. The page counts, medians, upper percentiles, and missing-position count show whether a few large pages dominate the data. Rates use page counts, and click-through rate is clicks divided by impressions.

## Load the May and June data

I keep the large daily records in DuckDB and bring only page-level results into pandas. The first run saves the results locally. The daily batches keep memory use down.


In [8]:
import getpass
import os
import tempfile
from datetime import date, timedelta
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

FEATURE_COLUMNS = (
    'log_prior30_impressions', 'log_prior30_clicks', 'prior30_ctr',
    'prior30_avg_position_filled', 'position_missing',
)

LABEL_END = date(2026, 6, 30)
HISTORICAL_CUTOFF = LABEL_END - timedelta(days=30)
FEATURE_START = HISTORICAL_CUTOFF - timedelta(days=29)
CURRENT_START = HISTORICAL_CUTOFF + timedelta(days=1)

def repo_root() -> Path:
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / 'work' / 'notebooks').is_dir():
            return path
    raise FileNotFoundError('Open the repository or a folder inside it before running the notebook')

def _quote_path(path: Path) -> str:
    return str(path).replace('\\', '/').replace("'", "''")

def _feature_select(table: str, labeled: bool) -> str:
    extra = (
        ", future30_impressions, CASE WHEN future30_impressions < 0.8 * prior30_impressions "
        "THEN 1 ELSE 0 END AS future_decline_label"
    ) if labeled else ''
    return f"""
        SELECT client_hash_id, content_hash_id, prior30_impressions, prior30_clicks,
               prior30_avg_position,
               LN(1 + prior30_impressions) AS log_prior30_impressions,
               LN(1 + prior30_clicks) AS log_prior30_clicks,
               CAST(prior30_clicks AS DOUBLE) / prior30_impressions AS prior30_ctr,
               COALESCE(prior30_avg_position, 0.0) AS prior30_avg_position_filled,
               CASE WHEN prior30_avg_position IS NULL THEN 1 ELSE 0 END AS position_missing
               {extra}
        FROM {table}
    """

repo = repo_root()


In [9]:
def _summarize_month(con: duckdb.DuckDBPyConnection, source: str,
                     start: date, end: date, folder: Path) -> Path:
    """Stream a month to disk, then deduplicate one day at a time."""
    folder.mkdir()
    stage = folder / 'daily_source.parquet'
    days = folder / 'days'
    days.mkdir()
    summary = folder / 'summary.parquet'
    con.execute(f"""
        COPY (SELECT report_date, client_hash_id, content_hash_id,
                     gsc_impressions, gsc_clicks, gsc_avg_position
              FROM {source}
              WHERE report_date BETWEEN DATE '{start}' AND DATE '{end}')
        TO '{_quote_path(stage)}' (FORMAT PARQUET)
    """)
    day = start
    while day <= end:
        day_path = days / f'{day}.parquet'
        con.execute(f"""
            COPY (
                SELECT client_hash_id, content_hash_id,
                       MIN(gsc_impressions) AS impressions,
                       MIN(gsc_clicks) AS clicks,
                       CASE WHEN MIN(gsc_avg_position) > 0 AND MIN(gsc_impressions) > 0
                            THEN MIN(gsc_avg_position) * MIN(gsc_impressions)
                            ELSE 0 END AS position_numerator,
                       CASE WHEN MIN(gsc_avg_position) > 0 AND MIN(gsc_impressions) > 0
                            THEN MIN(gsc_impressions) ELSE 0 END AS position_denominator,
                       COUNT(*) - 1 AS repeated_rows,
                       CASE WHEN MIN(gsc_impressions) IS DISTINCT FROM MAX(gsc_impressions)
                                      OR MIN(gsc_clicks) IS DISTINCT FROM MAX(gsc_clicks)
                                      OR MIN(gsc_avg_position) IS DISTINCT FROM MAX(gsc_avg_position)
                                      OR (COUNT(gsc_impressions) > 0 AND COUNT(gsc_impressions) < COUNT(*))
                                      OR (COUNT(gsc_clicks) > 0 AND COUNT(gsc_clicks) < COUNT(*))
                                      OR (COUNT(gsc_avg_position) > 0 AND COUNT(gsc_avg_position) < COUNT(*))
                            THEN 1 ELSE 0 END AS conflicting_keys
                FROM read_parquet('{_quote_path(stage)}')
                WHERE report_date = DATE '{day}'
                GROUP BY client_hash_id, content_hash_id
            ) TO '{_quote_path(day_path)}' (FORMAT PARQUET)
        """)
        day += timedelta(days=1)
    stage.unlink()
    con.execute(f"""
        COPY (
            SELECT client_hash_id, content_hash_id, COUNT(*) AS observed_days,
                   COUNT(*) FILTER (WHERE impressions IS NULL OR clicks IS NULL) AS missing_days,
                   SUM(impressions) AS impressions, SUM(clicks) AS clicks,
                   SUM(position_numerator) / NULLIF(SUM(position_denominator), 0) AS avg_position,
                   SUM(repeated_rows) AS repeated_rows,
                   SUM(conflicting_keys) AS conflicting_keys
            FROM read_parquet('{_quote_path(days / '*.parquet')}')
            GROUP BY client_hash_id, content_hash_id
        ) TO '{_quote_path(summary)}' (FORMAT PARQUET)
    """)
    for day_path in days.glob('*.parquet'):
        day_path.unlink()
    days.rmdir()
    return summary


In [10]:
def build_frames(source_override: str | None = None,
                 connection: duckdb.DuckDBPyConnection | None = None
                 ) -> tuple[pd.DataFrame, pd.DataFrame]:
    output = repo_root() / 'work' / 'outputs'
    output.mkdir(parents=True, exist_ok=True)
    con = connection or duckdb.connect()
    with tempfile.TemporaryDirectory(prefix='lane2_build_', dir=output) as scratch:
        temp = Path(scratch)
        spill = output / 'lane2_duckdb_spill'
        spill.mkdir(exist_ok=True)
        con.execute(f"SET temp_directory = '{_quote_path(spill)}'")
        con.execute("SET memory_limit = '1GB'")
        con.execute('SET threads = 2')
        if source_override is None:
            token = os.environ.get('HF_TOKEN') or getpass.getpass('Hugging Face READ token: ')
            con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '"
                        + token.replace("'", "''") + "')")
            del token
            root = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
            may_source = f"read_parquet('{root}/month=2026-05/*.parquet')"
            june_source = f"read_parquet('{root}/month=2026-06/*.parquet')"
        else:
            may_source = june_source = source_override

        may = _summarize_month(con, may_source, FEATURE_START, HISTORICAL_CUTOFF,
                               temp / 'may')
        june = _summarize_month(con, june_source, CURRENT_START, LABEL_END,
                                temp / 'june')
        may_rel = f"read_parquet('{_quote_path(may)}')"
        june_rel = f"read_parquet('{_quote_path(june)}')"
        counts = con.sql(f"""
            SELECT SUM(repeated_rows), SUM(conflicting_keys),
                   COUNT(*) FILTER (WHERE client_hash_id IS NULL OR content_hash_id IS NULL)
            FROM (SELECT * FROM {may_rel} UNION ALL SELECT * FROM {june_rel})
        """).fetchone()
        repeated, conflicting, missing_keys = (int(value or 0) for value in counts)
        assert missing_keys == 0, f'{missing_keys:,} rows have a missing daily key'
        assert conflicting == 0, f'{conflicting:,} daily keys have different GSC values'

        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE historical_metrics AS
            SELECT m.client_hash_id, m.content_hash_id,
                   m.impressions AS prior30_impressions,
                   m.clicks AS prior30_clicks,
                   m.avg_position AS prior30_avg_position,
                   j.impressions AS future30_impressions
            FROM {may_rel} m
            JOIN {june_rel} j USING (client_hash_id, content_hash_id)
            WHERE m.observed_days = 30 AND m.missing_days = 0
              AND m.impressions >= 100
              AND j.observed_days = 30 AND j.missing_days = 0
        """)
        con.execute(f"""
            CREATE OR REPLACE TEMP TABLE current_metrics AS
            SELECT client_hash_id, content_hash_id,
                   impressions AS prior30_impressions,
                   clicks AS prior30_clicks,
                   avg_position AS prior30_avg_position
            FROM {june_rel}
            WHERE observed_days = 30 AND missing_days = 0 AND impressions >= 100
        """)
        historical = con.sql(_feature_select('historical_metrics', labeled=True)).df()
        current = con.sql(_feature_select('current_metrics', labeled=False)).df()

    assert len(historical) > 0 and len(current) > 0, 'No pages met the coverage and impression rules'
    assert historical[list(FEATURE_COLUMNS)].notna().all().all()
    assert current[list(FEATURE_COLUMNS)].notna().all().all()
    print(f'May-June repeated GSC rows removed: {repeated:,}')
    print(f'Historical labeled pages: {len(historical):,}; current June pages: {len(current):,}')
    return historical, current

def load_frames() -> tuple[pd.DataFrame, pd.DataFrame]:
    output = repo_root() / 'work' / 'outputs'
    history_path = output / 'lane2_history_2026-05-31.parquet'
    current_path = output / 'lane2_current_2026-06-30.parquet'
    if history_path.exists() and current_path.exists():
        con = duckdb.connect()
        history = con.sql(f"SELECT * FROM read_parquet('{_quote_path(history_path)}')").df()
        current = con.sql(f"SELECT * FROM read_parquet('{_quote_path(current_path)}')").df()
        print(f'Loaded cached frames: {len(history):,} historical, {len(current):,} current pages')
        return history, current
    history, current = build_frames()
    output.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect()
    con.register('cache_history', history)
    con.register('cache_current', current)
    con.execute(f"COPY cache_history TO '{_quote_path(history_path)}' (FORMAT PARQUET)")
    con.execute(f"COPY cache_current TO '{_quote_path(current_path)}' (FORMAT PARQUET)")
    print('Saved local Parquet caches under work/outputs; dataset files are gitignored')
    return history, current


In [11]:
from pathlib import Path
import numpy as np
import pandas as pd

history, current = load_frames()
print(f'Eligible historical pages: {len(history):,}; clients: {history.client_hash_id.nunique():,}')
print(f'Observed June decline rate: {history.future_decline_label.mean():.1%}')
for name in ('prior30_impressions', 'prior30_clicks', 'prior30_ctr', 'prior30_avg_position'):
    values = history[name].dropna()
    p50, p90, p99 = values.quantile([0.50, 0.90, 0.99])
    print(f'{name}: n={len(values):,}, median={p50:.3g}, p90={p90:.3g}, p99={p99:.3g}')
print(f'Pages with missing May average position: {history.position_missing.sum():,}')

Loaded cached frames: 102,532 historical, 96,678 current pages
Eligible historical pages: 102,532; clients: 42
Observed June decline rate: 65.8%
prior30_impressions: n=102,532, median=606, p90=5.28e+03, p99=2.87e+04
prior30_clicks: n=102,532, median=1, p90=18, p99=110
prior30_ctr: n=102,532, median=0.0017, p90=0.00833, p99=0.0205
prior30_avg_position: n=102,532, median=13.8, p90=41.8, p99=71.9
Pages with missing May average position: 0


## 2. Three signal tests

I compare low and high quarters of May CTR, average position, and clicks against the observed June decline. I call a pattern directional when the overall difference points the expected way and at least 60% of comparable clients show the same direction. A client needs at least 10 pages in each group, and I need at least five comparable clients. This is a descriptive check on one time window, not proof that editing a page would change traffic.


In [12]:
def signal_test(frame, field, question, low_expected_higher, floor=50):
    data = frame[['client_hash_id', field, 'future_decline_label']].dropna()
    q25, q75 = data[field].quantile([0.25, 0.75])
    if q25 >= q75:
        print(f'{question}: INSUFFICIENT variation')
        return 'INSUFFICIENT'
    low = data.loc[data[field] <= q25]
    high = data.loc[data[field] >= q75]
    if min(len(low), len(high)) < floor:
        print(f'{question}: INSUFFICIENT pages (low n={len(low)}, high n={len(high)})')
        return 'INSUFFICIENT'

    low_rate = low.future_decline_label.mean()
    high_rate = high.future_decline_label.mean()
    low_clients = low.groupby('client_hash_id').future_decline_label.agg(['mean', 'count'])
    high_clients = high.groupby('client_hash_id').future_decline_label.agg(['mean', 'count'])
    clients = low_clients.join(high_clients, lsuffix='_low', rsuffix='_high')
    clients = clients.loc[(clients.count_low >= 10) & (clients.count_high >= 10)]
    if len(clients) < 5:
        print(f'{question}: INSUFFICIENT comparable clients ({len(clients)})')
        return 'INSUFFICIENT'

    if low_expected_higher:
        same_direction = clients.mean_low > clients.mean_high
        opposite_direction = clients.mean_low < clients.mean_high
        overall_gap = low_rate - high_rate
    else:
        same_direction = clients.mean_high > clients.mean_low
        opposite_direction = clients.mean_high < clients.mean_low
        overall_gap = high_rate - low_rate
    if overall_gap > 0 and same_direction.mean() >= 0.60:
        verdict = 'DIRECTIONAL'
    elif overall_gap < 0 and opposite_direction.mean() >= 0.60:
        verdict = 'OPPOSITE'
    else:
        verdict = 'MIXED'
    print(f'{question}: low n={len(low):,}, decline={low_rate:.1%}; '
          f'high n={len(high):,}, decline={high_rate:.1%}; '
          f'same direction in {int(same_direction.sum())}/{len(clients)} clients; '
          f'verdict={verdict}')
    return verdict

signal_results = {
    'low CTR': signal_test(history, 'prior30_ctr', 'Low CTR and later decline', True),
    'worse position': signal_test(history, 'prior30_avg_position',
                                  'Worse position and later decline', False),
    'fewer clicks': signal_test(history, 'prior30_clicks',
                                'Fewer clicks and later decline', True),
}

Low CTR and later decline: low n=37,424, decline=71.0%; high n=25,666, decline=54.9%; same direction in 26/30 clients; verdict=DIRECTIONAL
Worse position and later decline: low n=25,633, decline=57.8%; high n=25,633, decline=69.4%; same direction in 17/25 clients; verdict=DIRECTIONAL
Fewer clicks and later decline: low n=37,424, decline=71.0%; high n=26,285, decline=60.2%; same direction in 22/26 clients; verdict=DIRECTIONAL


## 3. Check the low-CTR flag assumption

A click-through-rate fix flag assumes that a visible page with weak click capture deserves a closer look. I test the observable part of that idea without using FlyRank's flag itself. Among pages above the median May impression count, the check compares low-CTR and high-CTR groups on the June decline label. It does not test whether changing a title would improve traffic.

In [13]:
visibility_floor = history.prior30_impressions.median()
visible_pages = history.loc[history.prior30_impressions >= visibility_floor]
print(f'Visible-page slice: {len(visible_pages):,} pages; '
      f'May impressions at least {visibility_floor:,.0f}')
flag_assumption_verdict = signal_test(
    visible_pages, 'prior30_ctr', 'Low CTR among visible pages', True
)
print('The product flag itself was not used as a feature or label')

Visible-page slice: 51,279 pages; May impressions at least 606
Low CTR among visible pages: low n=12,821, decline=78.4%; high n=12,820, decline=54.8%; same direction in 22/22 clients; verdict=DIRECTIONAL
The product flag itself was not used as a feature or label


## 4. What this means in practice

These observed patterns can help decide which pages deserve a closer look. A directional result supports a review question; a mixed or opposite result weakens it. The June label measures a later impression change. It does not show that a refresh would help.


In [14]:
for name, verdict in signal_results.items():
    print(f'{name}: {verdict}')
print(f'Visible low-CTR check: {flag_assumption_verdict}')
print('Use these results as directional review evidence, not as an automatic action rule')

low CTR: DIRECTIONAL
worse position: DIRECTIONAL
fewer clicks: DIRECTIONAL
Visible low-CTR check: DIRECTIONAL
Use these results as directional review evidence, not as an automatic action rule


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.